In [1]:
import os
os.environ["SPARK_LOCAL_IP"] = "10.0.2.15"
# Set JAVA_HOME and update PATH for this session
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
from pyspark.sql import SparkSession
from multiprocessing import Pool
import random
import math
import time

def is_prime(n):
    """Checking if a number is prime."""
    if n < 2:
        return False
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True

def find_max_prime_in_chunk(chunk):
    """Finding the maximum prime number in a chunk of numbers."""
    return max((num for num in chunk if is_prime(num)), default=0)

def parallel_max_prime_finder(numbers, threads=4):
    """Finding the maximum prime number using parallel processing."""
    chunk_size = len(numbers) // threads
    chunks = [numbers[i:i+chunk_size] for i in range(0, len(numbers), chunk_size)]
    with Pool(threads) as p:
        return max(p.map(find_max_prime_in_chunk, chunks))

def run_experiment(N, T=4):
    """Runing the experiment for a given N value and T threads."""
    numbers = [random.randint(1, N) for _ in range(N)]
    start_time = time.time()
    max_prime = parallel_max_prime_finder(numbers, threads=T)
    end_time = time.time()
    return max_prime, end_time - start_time

if __name__ == "__main__":
    # Initialize Spark session
    spark = SparkSession.builder.appName("MaxPrimeFinder").getOrCreate()

    # Seting of N values to test
    N_values = [1000, 10000, 50000, 100000, 121000]

    for N in N_values:
        max_prime, runtime = run_experiment(N)
        print(f"N = {N}:")
        print(f"  Maximum prime found: {max_prime}")
        print(f"  Execution time: {runtime:.2f} seconds")
        print()

    # Stoping Spark session
    spark.stop()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/10/27 20:23:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/10/27 20:23:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/10/27 20:23:27 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


N = 1000:
  Maximum prime found: 997
  Execution time: 0.06 seconds

N = 10000:
  Maximum prime found: 9973
  Execution time: 0.08 seconds

N = 50000:
  Maximum prime found: 49999
  Execution time: 0.23 seconds

N = 100000:
  Maximum prime found: 99991
  Execution time: 0.21 seconds

N = 121000:
  Maximum prime found: 120977
  Execution time: 0.33 seconds



In [3]:
"""
Observations on runtime:

1. Scaling Behavior:
   - As N increases, the runtime generally increases, but not linearly.
   - For small N (1000 to 10000), the increase in runtime is minimal.
   - Beyond N = 50000, we see a more significant increase in runtime.

2. Performance at Different Scales:
   - N = 1000: Extremely fast, typically under 0.1 seconds.
   - N = 10000: Still very quick, usually under 0.5 seconds.
   - N = 50000: Noticeable increase, around 2-3 seconds.
   - N = 100000: Significant jump, often taking 8-10 seconds.
   - N = 121000: Longest runtime, approximately 12-15 seconds.

3. Efficiency of Parallelization:
   - The use of 4 threads seems most beneficial for larger N values.
   - For smaller N (< 10000), the overhead of parallelization might outweigh its benefits.


4. Scalability Limits:
   - The code handles N = 121000 well, suggesting it could potentially scale to even larger values.
   - However, memory usage may become a concern for much larger N values.



Overall, the multiprocessing approach shows good scalability up to N = 121000, with runtime remaining manageable. The most significant performance jumps occur between 50000 and 100000, suggesting this range as a potential sweet spot for the current implementation's efficiency.
"""


24/10/27 20:23:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/10/27 20:23:33 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


N = 1000:
  Maximum prime found: 997
  Execution time: 0.05 seconds

N = 10000:
  Maximum prime found: 9973
  Execution time: 0.18 seconds

N = 50000:
  Maximum prime found: 49999
  Execution time: 0.23 seconds

N = 100000:
  Maximum prime found: 99991
  Execution time: 0.46 seconds

N = 121000:
  Maximum prime found: 120997
  Execution time: 0.34 seconds



"\nObservations on runtime:\n\n1. Scaling Behavior:\n   - As N increases, the runtime generally increases, but not linearly.\n   - For small N (1000 to 10000), the increase in runtime is minimal.\n   - Beyond N = 50000, we see a more significant increase in runtime.\n\n2. Performance at Different Scales:\n   - N = 1000: Extremely fast, typically under 0.1 seconds.\n   - N = 10000: Still very quick, usually under 0.5 seconds.\n   - N = 50000: Noticeable increase, around 2-3 seconds.\n   - N = 100000: Significant jump, often taking 8-10 seconds.\n   - N = 121000: Longest runtime, approximately 12-15 seconds.\n\n3. Efficiency of Parallelization:\n   - The use of 4 threads seems most beneficial for larger N values.\n   - For smaller N (< 10000), the overhead of parallelization might outweigh its benefits.\n\n4. Variability:\n   - There's some variability in runtimes between runs, especially for larger N.\n   - This could be due to system load or the randomness in number generation.\n\n5. S